<a href="https://colab.research.google.com/github/indtajith20053/norlax-ai-business-assistant/blob/main/norlax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q openai

In [ ]:
from openai import OpenAI
from getpass import getpass

# Paste your actual Groq API key when prompted in the input box
api_key = getpass("Enter your Groq API Key: ")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "qwen/qwen3.8-27b\"

print("Client configured!")

Enter your Groq API Key: ··········
Client configured!


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Say hello in one short sentence."
        }
    ]
)

print(response.choices[0].message.content)

Hello!


In [ ]:
BUSINESS_NAME = "Norlax AI Solutions"

PRODUCTS = {
    "business website": {"price": 45000, "stock": 5},
    "ai chatbot": {"price": 25000, "stock": 4},
    "n8n automation": {"price": 30000, "stock": 3},
    "pos system": {"price": 65000, "stock": 2},
    "ai business assistant": {"price": 50000, "stock": 3},
}

print("Business:", BUSINESS_NAME)
print("Number of products:", len(PRODUCTS))

Business: Norlax AI Solutions
Number of products: 5


In [ ]:
def get_product_info(product_name):
    product_name = product_name.lower().strip()

    if product_name in PRODUCTS:
        product = PRODUCTS[product_name]

        return (
            f"{product_name}: "
            f"price LKR {product['price']:,}, "
            f"stock {product['stock']}"
        )

    return (
        f"Sorry, we don't sell '{product_name}'. "
        f"We offer: {', '.join(PRODUCTS.keys())}"
    )

In [ ]:
def calculate_total(product_name, quantity):
    product_name = product_name.lower().strip()

    if product_name not in PRODUCTS:
        return f"Sorry, we don't sell '{product_name}'."

    if not isinstance(quantity, int) or isinstance(quantity, bool):
        return "Quantity must be a whole number."

    if quantity <= 0:
        return "Quantity must be greater than zero."

    price = PRODUCTS[product_name]["price"]
    total = price * quantity

    return f"{quantity} x {product_name} = LKR {total:,}"

In [ ]:
print(get_product_info("ai chatbot"))
print(calculate_total("ai chatbot", 2))

ai chatbot: price LKR 25,000, stock 4
2 x ai chatbot = LKR 50,000


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_product_info",
            "description": (
                "Get the listed price and available project slots "
                "for a product or service."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "The exact name of the product or service."
                    }
                },
                "required": ["product_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_total",
            "description": (
                "Calculate the total price for a specified quantity "
                "of a product or service using its listed price."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "The name of the product or service."
                    },
                    "quantity": {
                        "type": "integer",
                        "description": "The number of units requested."
                    },
                },
                "required": ["product_name", "quantity"],
            },
        },
    },
]

available_functions = {
    "get_product_info": get_product_info,
    "calculate_total": calculate_total,
}

print("Tools configured:", list(available_functions.keys()))

Tools configured: ['get_product_info', 'calculate_total']


In [ ]:
SYSTEM_PROMPT = f"""
You are a friendly, professional sales assistant for {BUSINESS_NAME}.

Your responsibilities:
1. Help customers understand our products and services.
2. Use get_product_info to check prices and availability.
3. Use calculate_total when a customer requests a total price.
4. Never invent prices or availability information.
5. Display prices in Sri Lankan rupees (LKR).
6. Explain that prices are estimates until a formal quotation is approved.
7. Do not claim that a booking or purchase has been confirmed.
8. Politely explain when we do not offer a requested service.
9. Redirect unrelated questions to our business services.
10. Keep responses clear, concise, and professional.
"""

In [ ]:
import json


def run_agent(user_message):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    # Prevent an endless tool-calling loop.
    for _ in range(5):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        msg = response.choices[0].message

        # The model has finished answering.
        if not msg.tool_calls:
            return msg.content or "Sorry, I couldn't generate a response."

        # Add the assistant's tool-call request to the conversation.
        messages.append(msg)

        # Execute the requested Python functions.
        for call in msg.tool_calls:
            function_name = call.function.name
            arguments = json.loads(call.function.arguments)

            if function_name not in available_functions:
                result = "Error: Unknown tool requested."
            else:
                try:
                    result = available_functions[function_name](**arguments)
                except (TypeError, ValueError):
                    result = "Error: Invalid tool arguments."

            print(
                f"🔧 Tool used: {function_name}"
                f"({arguments}) → {result}"
            )

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })

    return "Sorry, I couldn't complete the request."

In [ ]:
print(run_agent("How much is an AI chatbot?"))

🔧 Tool used: get_product_info({'product_name': 'AI Chatbot'}) → ai chatbot: price LKR 25,000, stock 4
Our AI Chatbot service is listed at **LKR 25,000**.

Please note:
- This is an estimate until a formal quotation is approved.
- There are currently **4 project slots** available.

Would you like me to calculate a total for a specific quantity, or answer any questions about the service?


In [ ]:
# Test 1: Product information
print("TEST 1")
print(run_agent("How much is an AI chatbot?"))

# Test 2: Calculate total
print("\nTEST 2")
print(run_agent("I need 2 AI chatbots. What is the total?"))

# Test 3: Unknown product
print("\nTEST 3")
print(run_agent("Do you sell chocolate cake?"))

# Test 4: Unrelated question
print("\nTEST 4")
print(run_agent("Who won the cricket match?"))

# Test 5: Multiple products
print("\nTEST 5")
print(run_agent("What would 3 business websites cost?"))

TEST 1
🔧 Tool used: get_product_info({'product_name': 'AI chatbot'}) → ai chatbot: price LKR 25,000, stock 4
Our AI chatbot is priced at **LKR 25,000** per unit.

Please note:
- This price is an estimate until a formal quotation is approved.
- There are currently **4 project slots** available.

Would you like me to calculate a total for a specific quantity, or do you have any other questions about the service?

TEST 2
🔧 Tool used: get_product_info({'product_name': 'AI Chatbot'}) → ai chatbot: price LKR 25,000, stock 4
🔧 Tool used: calculate_total({'product_name': 'AI Chatbot', 'quantity': 2}) → 2 x ai chatbot = LKR 50,000
The price for 2 AI chatbots is **LKR 50,000**.

Please note that this is an estimate until a formal quotation is approved. We currently have 4 units available. Would you like to proceed with a formal quote?

TEST 3
I specialize in AI solutions and business services, so I do not sell chocolate cake or other food products.

How can I help you with our technology solutio